# Практика: доли цифр и разбиение выборки

Делим таблицу на **обучающую** и **проверочную** части и следим, чтобы доли цифр не разъехались.

In [ ]:
from pathlib import Path
import pandas as pd


DATA_URL = (
    "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/modules/08_04_mnist_knn/data/digits.csv"
)


def find_digits_csv():
    for p in (
        Path("digits.csv"),
        Path("../digits.csv"),
        Path("../../data/digits.csv"),
        Path("../data/digits.csv"),
        Path("../../../data/digits.csv"),
    ):
        if p.exists():
            return p.resolve()
    return DATA_URL


DIGITS_PATH = find_digits_csv()
df = pd.read_csv(DIGITS_PATH)
PIXELS = [c for c in df.columns if c.startswith('p')]


## 1. Разбиение руками

Перемешайте строки: `order = df.sample(frac=1, random_state=0).index`. Первые 75% номеров -> `train_idx`, остальные -> `test_idx`.

**Зачем seed:** без `random_state` каждый запуск даёт другое разбиение и другие числа в отчёте.

In [ ]:
order = None
train_idx = None
test_idx = None
assert order is not None and train_idx is not None and test_idx is not None
assert len(train_idx) == 1347 and len(test_idx) == 450
print(len(train_idx), len(test_idx))

## 2. Части не пересекаются

Проверьте, что ни один номер не попал в обе части: `n_overlap` — сколько номеров из `train_idx` встречается в `test_idx`.

**How:** `train_idx.isin(test_idx).sum()`.

In [ ]:
n_overlap = None
n_total = None
assert n_overlap is not None and int(n_overlap) == 0
assert n_total is not None and int(n_total) == len(df)
print(n_overlap, n_total)

## 3. Доли цифр в двух частях

`train_df` и `test_df` — строки по этим номерам (`df.loc[...]`). Посчитайте доли цифр в каждой части и максимальное расхождение по цифрам -> `max_gap`.

**How:** две `value_counts(normalize=True).sort_index()`, затем `(a - b).abs().max()`.

In [ ]:
train_df = None
test_df = None
max_gap = None
assert train_df is not None and test_df is not None
assert len(train_df) + len(test_df) == len(df)
assert max_gap is not None and 0 < float(max_gap) < 0.1
print(round(float(max_gap), 4))

## 4. То же самое библиотекой

`train_test_split` из модуля 2: `test_size=0.25`, `random_state=0`. Запишите размеры -> `n_tr_lib`, `n_te_lib` и убедитесь, что они совпали с ручным разбиением.

In [ ]:
from sklearn.model_selection import train_test_split

n_tr_lib = None
n_te_lib = None
assert n_tr_lib == len(train_idx) and n_te_lib == len(test_idx)
print(n_tr_lib, n_te_lib)

## 5. Разбиение с сохранением долей

Тот же вызов с `stratify=df['label']`. Посчитайте расхождение долей -> `max_gap_strat` и сравните с `max_gap` из блока 3.

**Вопрос:** зачем сохранять доли, если разбиение и так случайное?

In [ ]:
max_gap_strat = None
assert max_gap_strat is not None
assert float(max_gap_strat) < float(max_gap)
print(round(float(max_gap_strat), 4), round(float(max_gap), 4))

## 6. Доля цифры при условии

`n_dark` — сколько пикселей картинки ярче 8 (сколько «чернил»). Разделите таблицу на картинки с `n_dark` выше медианы и остальные.

Посчитайте долю цифры 8 в каждой части -> `p_eight_dark`, `p_eight_light`.

В `COND_NOTE` объясните результат: чем восьмёрка отличается от других цифр по чернилам и как это использует поиск ближайших соседей.

In [ ]:
n_dark = (df[PIXELS] > 8).sum(axis=1)
p_eight_dark = None
p_eight_light = None
COND_NOTE = ''
assert p_eight_dark is not None and p_eight_light is not None
assert float(p_eight_dark) > 1.5 * float(p_eight_light)
assert len(COND_NOTE) > 40
print(round(float(p_eight_dark), 3), round(float(p_eight_light), 3), COND_NOTE)

## 7. Проверочная часть — не для подглядывания

Возьмите крошечную проверочную часть: `tiny = df.sample(20, random_state=11)`. Доля цифры 3 в ней -> `p_three_tiny`; доля в полной таблице -> `p_three_all`.

Запишите в `TINY_NOTE`, почему по 20 картинкам нельзя судить о качестве распознавателя.

In [ ]:
tiny = None
p_three_tiny = None
p_three_all = None
TINY_NOTE = ''
assert tiny is not None and len(tiny) == 20
assert p_three_tiny is not None and p_three_all is not None
assert len(TINY_NOTE) > 50
print(round(float(p_three_tiny), 3), round(float(p_three_all), 3))

## 8. Эксперимент: пять разных разбиений

Для `random_state` 0…4 посчитайте долю цифры 3 в проверочной части (25%) -> список `spreads` (5 чисел). Найдите разницу между максимумом и минимумом -> `spread_range`.

Вывод в `SEED_NOTE`: что это значит для сравнения двух распознавателей. **Готового ответа нет.**

In [ ]:
spreads = []
spread_range = None
SEED_NOTE = ''
assert len(spreads) == 5
assert spread_range is not None and float(spread_range) > 0
assert len(SEED_NOTE) > 50
print([round(float(s), 3) for s in spreads], round(float(spread_range), 3))

## 9. Расширение: две проверочные части

Разбейте таблицу на **три** части: 60% / 20% / 20% (обучение, проверка, финал). Запишите размеры -> `sizes_three` (список из трёх чисел, сумма = 1797).

На паре 28 эта третья часть понадобится, чтобы честно выбрать число соседей.

In [ ]:
sizes_three = []
assert len(sizes_three) == 3
assert sum(int(s) for s in sizes_three) == len(df)
assert min(int(s) for s in sizes_three) > 300
print(sizes_three)